# Test Reranker Approach

**Your friend's recommendation**: Reranker + Pure Cosine Similarity

This notebook tests the reranker approach with:
- Jina embeddings v3
- Pure cosine similarity 
- Arabic reranker for refinement

Focus: Testing the reranker approach only.

In [1]:
import os
import sys
import time
import pathlib

sys.path.append('/app')
os.chdir('/app')

HF_CACHE = pathlib.Path("models_cache")
HF_CACHE.mkdir(exist_ok=True)
os.environ["HF_HOME"] = str(HF_CACHE)

# Import what we need
from src.utils import ServiceDatasetLoader
from src.processing import TextNormalizer, MorphReducer, LexicalRelevanceFilter
from src.search import ServiceSearch
from src.presets import get_config, FRIEND_RECOMMENDED

import logging
logging.basicConfig(level=logging.INFO)
log = logging.getLogger("reranker-test")

print("✅ Ready to test reranker approach")

✅ Ready to test reranker approach


In [2]:
# Get reranker configuration
config = get_config(FRIEND_RECOMMENDED)

print("🎯 Reranker approach configuration:")
print(f"📝 {config['description']}")
print(f"   Embedding: {config['embedding_model']['ar']} (Arabic)")
print(f"   Similarity: {config['similarity']['ar']}")
print(f"   Lexical filtering: {config['search']['use_lexical_filtering']}")

🎯 Reranker approach configuration:
📝 Pure reranker: Jina embeddings + cosine similarity + Arabic reranker
   Embedding: all-MiniLM-L6-v2 (Arabic)
   Similarity: cosine-reranker
   Lexical filtering: False


In [3]:
# Initialize system
start_time = time.time()

normalizer = TextNormalizer()
morpher = MorphReducer()  # Now using PyArabic (fast, offline)
lexical_filter = LexicalRelevanceFilter(normalizer)

# Create loaders with proper parameters
loaders = {}
for lang, cfg in config["data"].items():
    loader = ServiceDatasetLoader(
        cfg["path"],
        rename_map=cfg["rename_map"],
        combine_cols=cfg["combine_cols"]
    )
    loaders[lang] = loader

engine = ServiceSearch(
    loaders,
    config,
    normalizer=normalizer,
    morpher=morpher,
    lexical_filter=lexical_filter
)

init_time = time.time() - start_time
log.info(f"🚀 Reranker system ready ({init_time:.2f}s) with PyArabic integration")

INFO:src.processing:🚀 PyArabic morphological reducer ready (fast, offline)
INFO:src.processing:📖 Loaded lexical config: 89 Arabic species, 48 Arabic actions, 53 English actions
INFO:src.utils:📄 Loading [NaamaServiceIn full Details] 'NaamaServiceIn full Details.xlsx'…
INFO:src.utils:✅ 997 rows → (service, service, service, classification, sector, description_short, description, beneficiaries)
INFO:src.utils:📄 Loading [NaamaServiceIn full Details] 'NaamaServiceIn full Details.xlsx'…
INFO:src.utils:✅ 997 rows → (service, service, service, classification, sector, description_short, description, beneficiaries)
INFO:src.search:🚀 Initialising ServiceSearch …
INFO:src.search:✅ Ready
INFO:reranker-test:🚀 Reranker system ready (0.43s) with PyArabic integration


In [4]:
# Test problematic queries
test_queries = {
    "فاكهة": "Was getting 0/20 results",
    "احفر بير": "Was returning cleaning instead of construction",
    "القطط": "Was returning shipping instead of pets", 
    "بسة": "Colloquial Arabic cats",
    "feed sales": "Was returning fish farming",
    "تربية نحل": "Should work well (baseline)",
}

print("🧪 Testing with reranker approach...")
print("=" * 50)

results = {}
for query, desc in test_queries.items():
    print(f"\n🔍 '{query}' - {desc}")
    
    start_query = time.time()
    result = engine.search(query)
    query_time = time.time() - start_query
    
    results[query] = result
    kept = len(result['hits_kept'])
    total = kept + len(result['hits_rejected'])
    
    print(f"   📊 Results: {kept}/{total} ({query_time:.3f}s)")
    
    if result['hits_kept']:
        for i, hit in enumerate(result['hits_kept'][:3], 1):
            print(f"      {i}. {hit['final_pct']:5.1f}% - {hit['title'][:50]}")

print("\n✅ Reranker testing complete")

INFO:src.search:⏳ Loading embedder 'all-MiniLM-L6-v2' for [ar] …
INFO:sentence_transformers.SentenceTransformer:Use pytorch device_name: cpu
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: all-MiniLM-L6-v2


🧪 Testing with reranker approach...

🔍 'فاكهة' - Was getting 0/20 results


INFO:src.search:📂 Loading cached vectorstore: ar_cosine-reranker_all-MiniLM-L6-v2_6e90ab8a
INFO:src.search:🔍 [ar/cosine-reranker] 'فاكهة' → norm='فاكهه' → base='فاكهه'


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

INFO:src.search:   ✅  0.6198 ( 62.0%)  نقل ملكية ترخيص تشغيلي تربية وإنتاج جدات أمهات الدجاج البياض وتشغيل مفارخها
INFO:src.search:   ✅  0.6180 ( 61.8%)  نقل ملكية ترخيص تشغيلي تربية وإنتاج جدات أمهات الدجاج اللاحم وتشغيل مفارخها
INFO:src.search:   ✅  0.6036 ( 60.4%)  إلغاء شهادة اعتماد مصدر مائي (عقد توريد) لمصانع إنتاج مياه معبأة
INFO:src.search:   ✅  0.5452 ( 54.5%)  استعلام عن المعاملات
INFO:src.search:   ✅  0.5429 ( 54.3%)  تركيب البيوت الزراعية
INFO:src.search:   ✅  0.5422 ( 54.2%)  الاستيراد والتصدير للأسمدة
INFO:src.search:   ✅  0.5383 ( 53.8%)  حجز موعد
INFO:src.search:   ✅  0.5372 ( 53.7%)  طلب ترقيم الماشية
INFO:src.search:   ✅  0.5365 ( 53.6%)  تجديد ترخيص تشغيلي تشغيل أحواض تفريخ الأسماك في البحار
INFO:src.search:   ✅  0.5362 ( 53.6%)  اصدار رخصة شهادة سعودي جاب
INFO:src.search:   ✅  0.5357 ( 53.6%)  نقل ملكية ترخيص تشغيلي تشغيل أحواض تفريخ الأسماك في البحار
INFO:src.search:   ✅  0.5352 ( 53.5%)  تغيير نشاط ترخيص تشغيلي   زراعة النباتات العطرية والزهور ( الأزهار وبراعم الأ

...

INFO:src.search:   ✅  0.9023 ( 90.2%)  Public Interest Markets License
INFO:src.search:   ✅  0.8961 ( 89.6%)  Managing Public Interest Markets License
INFO:src.search:   ✅  0.8945 ( 89.5%)  Transforming to Organic Production Request Registration for Agricultural Marketers
INFO:src.search:   ✅  0.8944 ( 89.4%)  Cancel Calf Fattening License
INFO:src.search:   ✅  0.8942 ( 89.4%)  Transforming to Organic Production Request Registration for Beekeepers
INFO:src.search:   ✅  0.8939 ( 89.4%)  Convert a Paper License to an Electronic License for ‎Operating Slaughterhouses for Slaughtering and Preparing Poultry, Rabbits and Birds Meat
INFO:src.search:   ✅  0.8939 ( 89.4%)  Transfer of Sponsorship for Agricultural Labor Services
INFO:src.search:   ✅  0.8937 ( 89.4%)  Cancel Post harvest services License
INFO:src.search:   ✅  0.8930 ( 89.3%)  Change Business Activity ‎Operating Slaughterhouses for Slaughtering and Preparing Poultry, Rabbits and Birds Meat Operational License
INFO:src.search:   ✅ 

   📊 Results: 20/20 (28.189s)
      1.  90.2% - Public Interest Markets License
      2.  89.6% - Managing Public Interest Markets License
      3.  89.5% - Transforming to Organic Production Request Registr

🔍 'تربية نحل' - Should work well (baseline)


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

INFO:src.search:   ✅  0.6450 ( 64.5%)  إلغاء ترخيص منحل أفراد
INFO:src.search:   ✅  0.5597 ( 56.0%)  تركيب البيوت الزراعية
INFO:src.search:   ✅  0.5540 ( 55.4%)  حجز موعد
INFO:src.search:   ✅  0.5497 ( 55.0%)  استعلام عن المعاملات
INFO:src.search:   ✅  0.5492 ( 54.9%)  طلب ترقيم الإبل
INFO:src.search:   ✅  0.5478 ( 54.8%)  نقل ملكية ترخيص تشغيلي تشغيل المجازر لذبح وتهيئة لحوم المواشي
INFO:src.search:   ✅  0.5461 ( 54.6%)  تغيير نشاط ترخيص تشغيلي تربية السمان (الفري)
INFO:src.search:   ✅  0.5461 ( 54.6%)  تغيير نشاط ترخيص تشغيلي تربية الأرانب
INFO:src.search:   ✅  0.5454 ( 54.5%)  تجديد ترخيص تشغيلي تربية الأرانب
INFO:src.search:   ✅  0.5454 ( 54.5%)  تغيير نشاط ترخيص تشغيلي تربية الحمام
INFO:src.search:   ✅  0.5446 ( 54.5%)  تغيير نشاط ترخيص تشغيلي تربية النعام
INFO:src.search:   ✅  0.5445 ( 54.5%)  تغيير نشاط ترخيص تشغيلي  زراعة المانجو
INFO:src.search:   ✅  0.5445 ( 54.4%)  تغيير نشاط ترخيص تشغيلي  زراعة البصل
INFO:src.search:   ✅  0.5442 ( 54.4%)  تغيير نشاط ترخيص تشغيلي زراعة التين

   📊 Results: 20/20 (4.372s)
      1.  64.5% - إلغاء ترخيص منحل أفراد
      2.  56.0% - تركيب البيوت الزراعية
      3.  55.4% - حجز موعد

✅ Reranker testing complete


In [5]:
# Summary
print("\n📊 RERANKER APPROACH SUMMARY")
print("=" * 40)

successful = sum(1 for r in results.values() if len(r['hits_kept']) > 0)
avg_results = sum(len(r['hits_kept']) for r in results.values()) / len(results)
good_results = sum(1 for r in results.values() if len(r['hits_kept']) >= 10)

print(f"Successful queries: {successful}/{len(test_queries)}")
print(f"Average results per query: {avg_results:.1f}")
print(f"Queries with 10+ results: {good_results}/{len(test_queries)}")

print("\n🎯 Key observations:")
print("• How do results look for previously problematic queries?")
print("• Are false positives reduced (احفر بير, القطط)?")
print("• Is performance acceptable?")
print("• Ready for comparison with original approach?")


📊 RERANKER APPROACH SUMMARY
Successful queries: 6/6
Average results per query: 20.0
Queries with 10+ results: 6/6

🎯 Key observations:
• How do results look for previously problematic queries?
• Are false positives reduced (احفر بير, القطط)?
• Is performance acceptable?
• Ready for comparison with original approach?
